## Imports and configuration

In [10]:
import numpy as np
import pandas as pd
from pyproj import Transformer
from scipy.spatial import cKDTree
from pathlib import Path

BASE = Path.cwd()
DATA_DIR = BASE / Path("../data")


## Postal Codes

Load raw GeoNames data (`PL.txt`), filter for Kraków and surrounding areas, and compute the centroid (mean coordinates) for each unique postal code along with its WKT point geometry.

In [11]:
column_names = [
    "country_code", "postal_code", "place_name", "admin_name1",
    "admin_code1", "admin_name2", "admin_code2", "admin_name3",
    "admin_code3", "latitude", "longitude", "accuracy"
]


df_raw = pd.read_csv(
    DATA_DIR / "PL.txt",
    sep="\t",
    header=None,
    names=column_names,
    dtype=str,
    encoding="utf-8"
)

df_raw["latitude"] = pd.to_numeric(df_raw["latitude"], errors="coerce")
df_raw["longitude"] = pd.to_numeric(df_raw["longitude"], errors="coerce")


krakow_filter = (df_raw["place_name"].str.lower() == "kraków") | (
    df_raw["admin_name2"].str.lower().isin(["kraków", "powiat krakowski"])
)
df_krakow = df_raw[krakow_filter].copy()


df_post_avg = (
    df_krakow.groupby("postal_code", as_index=False)
    .agg({
        "place_name": "first",
        "admin_name1": "first",
        "admin_name2": "first",
        "admin_name3": "first",
        "latitude": "mean",
        "longitude": "mean",
        "country_code": "count"
    })
    .rename(columns={"country_code": "punktow_w_kodzie"})
)

df_post_avg["point_wkt"] = df_post_avg.apply(
    lambda r: f"POINT({r['longitude']:.6f} {r['latitude']:.6f})", axis=1
)


path_post_avg = DATA_DIR / "krakow_postal_codes.csv"
df_post_avg.to_csv(path_post_avg, index=False, encoding="utf-8-sig")

display(df_post_avg.head())


,postal_code,place_name,admin_name1,admin_name2,admin_name3,latitude,longitude,punktow_w_kodzie,point_wkt
0,30-001,Kraków,Lesser Poland,Kraków,Kraków,50.0759,19.9410,1,POINT(19.941000 50.075900)
1,30-002,Kraków,Lesser Poland,Kraków,Kraków,50.0770,19.9364,1,POINT(19.936400 50.077000)
2,30-003,Kraków,Lesser Poland,Kraków,Kraków,50.0742,19.9324,1,POINT(19.932400 50.074200)
3,30-004,Kraków,Lesser Poland,Kraków,Kraków,50.0738,19.9349,1,POINT(19.934900 50.073800)
4,30-006,Kraków,Lesser Poland,Kraków,Kraków,50.0756,19.9314,1,POINT(19.931400 50.075600)


## Prices with postal codes

Reproject postal code coordinates to EPSG:2180 and use a KD-Tree spatial query to assign the nearest postal code and reference geometry to each apartment record.

In [12]:
df_apartments = pd.read_csv(DATA_DIR / "input_data.csv")
df_post = pd.read_csv(DATA_DIR / "krakow_postal_codes.csv", dtype={"postal_code": str})

transformer = Transformer.from_crs("EPSG:4326", "EPSG:2180", always_xy=True)
df_post["X_2180"], df_post["Y_2180"] = transformer.transform(
    df_post["longitude"].values, df_post["latitude"].values
)

post_coords = np.c_[df_post["X_2180"], df_post["Y_2180"]]
apt_coords = np.c_[df_apartments["X_2180"], df_apartments["Y_2180"]]

tree = cKDTree(post_coords)
_, indices = tree.query(apt_coords)

matched_post = df_post.iloc[indices]
df_apartments["postal_code"] = matched_post["postal_code"].values
df_apartments["postal_code_point_wkt"] = matched_post["point_wkt"].values

path_apartments = DATA_DIR / "apartments_with_postal_codes.csv"
df_apartments.to_csv(path_apartments, index=False, encoding="utf-8-sig")

display(df_apartments.head())

,cena_m2,powUzytkowaLokalu,nrKondygnacji,liczbaIzb,rodzajRynku,dzielnica,rok,dist_to_center,has_parking,has_balcony,has_storage,X_2180,Y_2180,postal_code,postal_code_point_wkt
0,16355.350027,36.94,3.0,2.0,Pierwotny,Dębniki,2025,3131.181316,0,0,0,565752.259295,241076.402356,30-364,POINT(19.916700 50.036800)
1,11508.544017,90.42,6.0,4.0,Pierwotny,Dębniki,2025,3096.293285,0,0,0,565793.520943,241104.123580,30-364,POINT(19.916700 50.036800)
2,20486.342438,30.02,1.0,1.0,Wtórny,Podgórze,2025,5189.112313,0,0,0,571168.965177,242225.302303,30-716,POINT(19.997300 50.042200)
3,9810.936052,43.16,5.0,2.0,Wtórny,Dębniki,2025,3131.181316,0,0,0,565752.259295,241076.402356,30-364,POINT(19.916700 50.036800)
4,20247.614135,38.77,8.0,2.0,Wtórny,Prądnik Biały,2025,3815.112605,1,1,0,565907.822944,247939.401488,31-226,POINT(19.925200 50.097300)


## Statistics

Calculate summary statistics (mean and median price per square meter) grouped by postal code and export the aggregated results to CSV.

In [13]:
price_stats = (
    df_apartments.groupby("postal_code")["cena_m2"]
    .agg(
        avg_price_m2="mean", 
        median_price_m2="median"
    )
    .round(2)
    .reset_index()
)


path_stats = DATA_DIR / "price_per_m2_statistics_postal_codes.csv"
price_stats.to_csv(path_stats, index=False, encoding="utf-8")

display(price_stats.head())

,postal_code,avg_price_m2,median_price_m2
0,30-001,18647.67,18647.67
1,30-002,18839.57,18614.87
2,30-003,21842.92,19900.00
3,30-004,16040.40,15024.70
4,30-006,17857.92,18289.41
